In [ ]:
# --- Hàm xây dựng Ma trận A (từ Ảnh 2) ---
def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

In [ ]:
# --- CENTRAL SOLVER ---
def solve_centralized(supply_s, capacity_b, valuations, budgets, A):
  """
  Solves the primal E-G program with personalized constraints
  and prints iteration logs.
  """
  n_buyers, m_goods = valuations.shape
  obj_hist = []

  # Shared variable
  X = cp.Variable((n_buyers, m_goods), nonneg=True)

  # Constraints
  constraints = [cp.sum(X, axis=0) <= supply_s]
  for i in range(n_buyers):
      constraints.append(A @ X[i] <= capacity_b[i])

  # Objective
  obj_terms = []
  utilities = cp.sum(cp.multiply(valuations, X), axis=1)
  primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

  objective = cp.Maximize(primal_utility)

  # Initial objective (lúc này chưa ai mua gì)
  p = np.ones(m_goods)
  q = np.ones((n_buyers, m_goods // 6))
  util = [valuations[j] @ np.ones(m_goods) for j in range(n_buyers)]
  term_p = np.sum(p * supply_s)     # p * Supply
  term_q = np.sum(q * capacity_b)            # q * Capacity
  term_u = np.sum(budgets * np.log(util) + 1e-12) # Utility
  obj_0 = term_p + term_q + term_u - np.sum(budgets)
  obj_hist.append(obj_0)

  # Solve problem
  prob = cp.Problem(objective, constraints)

  try:
      prob.solve(solver=cp.SCS, verbose=False)
  except cp.error.SolverError:
      print("SCS failed or not installed.")

  obj_hist.append(prob.value)
  stats = prob.solver_stats
  print(f"\nIterations: {stats.num_iters}")
  if prob.status not in ['optimal', 'optimal_inaccurate']:
      print(f"\nCentralized solver failed/infeasible. Status: {prob.status}")
      return None

  return X.value, obj_hist, stats.num_iters